# Detektor Pesan Penipuan SMS/WA Berbahasa Indonesia
**Pipeline end-to-end:** collect → preprocessing → feature engineering → training → evaluasi → deploy (bot Telegram)

Kelas: **0 = normal**, **1 = penipuan**, **2 = promo**

### Struktur folder
```
proyek/
├── detektor_penipuan_sms.ipynb   ← notebook ini
├── data/
│   └── dataset_sms_spam_v1.csv   ← unduh dari http://bit.ly/yw_sms_spam_indonesia
├── sms_utils.py                  ← dibuat otomatis oleh notebook (sel 1)
├── bot.py, requirements.txt      ← dibuat otomatis (bagian 13)
└── models/                       ← model hasil training
```

### Cara menjalankan di VS Code
1. Pasang ekstensi **Python** dan **Jupyter** di VS Code.
2. Buka folder proyek, lalu buat virtual environment di terminal:
   `python -m venv .venv`, kemudian aktifkan (`.venv\Scripts\activate` di Windows, `source .venv/bin/activate` di Linux/Mac).
3. Pasang paket: `pip install pandas numpy scipy scikit-learn matplotlib joblib ipykernel python-telegram-bot`
4. Di pojok kanan atas notebook, klik **Select Kernel** lalu pilih `.venv`.
5. Jalankan sel dari atas ke bawah (**Run All**).

**Sumber data:** Rahmi, F. and Wibisono, Y. (2016). *Aplikasi SMS Spam Filtering pada Android menggunakan Naive Bayes*, Unpublished manuscript. Lisensi data: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).

## 1. Setup
Sel di bawah menulis `sms_utils.py`, yaitu fungsi-fungsi yang dipakai **bersama** oleh notebook dan bot: normalisasi teks, fitur buatan, aturan threshold, dan penjelasan prediksi. Fungsi ini sengaja ditaruh di file terpisah, karena model `.joblib` menyimpan *referensi* ke fungsi tersebut. Kalau fungsinya didefinisikan di dalam notebook, bot tidak akan bisa me-load model.

In [ ]:
%%writefile sms_utils.py
"""Utilitas bersama untuk notebook dan bot Telegram detektor pesan penipuan.

File ini ditulis otomatis oleh notebook (sel %%writefile). Fungsi di sini
dipakai di dalam pipeline scikit-learn, jadi harus tetap ada di folder yang
sama dengan bot.py supaya model .joblib bisa di-load.
"""
import math
import re
import unicodedata

import numpy as np

# ---------------------------------------------------------------------------
# 1. Normalisasi teks
# ---------------------------------------------------------------------------
RE_URL = re.compile(
    r"(?:https?://|www\.)\S+"
    r"|\b[\w\-]+(?:\.[\w\-]+)*\.(?:com|net|org|id|xyz|site|website|online|info|link|top|"
    r"club|ly|me|co|biz|shop|store|live|icu|vip|cc)\b(?:/\S*)?",
    re.IGNORECASE,
)
RE_PHONE = re.compile(r"(?:\+62|\b62|\b0)\s?8\d{1,3}[\s\-]?\d{3,4}[\s\-]?\d{2,5}")
RE_USSD = re.compile(r"\*\d+(?:\*\d+)*#")
RE_APK = re.compile(r"\.apk\b|\bapk\b", re.IGNORECASE)
RE_NON_WORD = re.compile(r"[^\w\s]")
RE_REPEAT = re.compile(r"(\w)\1{2,}")
RE_SPACE = re.compile(r"\s+")

# Kamus singkatan/slang -> bentuk baku. Silakan ditambah sendiri.
SLANG = {
    "yg": "yang", "dgn": "dengan", "dg": "dengan", "utk": "untuk", "u": "untuk",
    "tdk": "tidak", "gak": "tidak", "ga": "tidak", "gk": "tidak", "nggak": "tidak", "ngga": "tidak",
    "sdh": "sudah", "udh": "sudah", "udah": "sudah", "blm": "belum", "bs": "bisa", "bsa": "bisa",
    "krn": "karena", "karna": "karena", "sy": "saya", "aq": "aku", "km": "kamu", "kmu": "kamu",
    "dr": "dari", "pd": "pada", "jg": "juga", "tsb": "tersebut", "slmt": "selamat",
    "dpt": "dapat", "dptkan": "dapatkan", "mendptkan": "mendapatkan", "hub": "hubungi",
    "tlp": "telepon", "telp": "telepon", "no": "nomor", "rek": "rekening", "tf": "transfer",
    "bgt": "banget", "aja": "saja", "lg": "lagi", "trs": "terus", "tp": "tapi", "kmrn": "kemarin",
    "skrg": "sekarang", "org": "orang", "bln": "bulan", "thn": "tahun", "hr": "hari",
    "mnt": "menit", "rb": "ribu", "jt": "juta", "infonya": "info",
}

# Token pengganti hasil masking (tanpa <> agar tidak dibuang tokenizer TF-IDF)
TOKEN_URL, TOKEN_PHONE, TOKEN_NUM, TOKEN_USSD = "urltoken", "nomortoken", "angkatoken", "ussdtoken"
TOKEN_LABEL = {
    TOKEN_URL: "[link]", TOKEN_PHONE: "[nomor HP]", TOKEN_NUM: "[angka]", TOKEN_USSD: "[kode USSD]",
}


def _mask_token(tok):
    """Token yang mayoritas angka (nominal, PIN, kode) -> angkatoken.
    Token 'leet' seperti h4diah atau nama1 dibiarkan karena itu sinyal."""
    digits = sum(c.isdigit() for c in tok)
    if digits and digits / len(tok) >= 0.5:
        return TOKEN_NUM
    return SLANG.get(tok, tok)


def normalize_text(text):
    t = unicodedata.normalize("NFKC", str(text)).lower()
    t = RE_URL.sub(f" {TOKEN_URL} ", t)
    t = RE_USSD.sub(f" {TOKEN_USSD} ", t)
    t = RE_PHONE.sub(f" {TOKEN_PHONE} ", t)
    t = RE_NON_WORD.sub(" ", t)
    t = RE_REPEAT.sub(r"\1\1", t)
    tokens = [_mask_token(tok) for tok in t.split()]
    return RE_SPACE.sub(" ", " ".join(tokens)).strip()


# ---------------------------------------------------------------------------
# 2. Fitur buatan sendiri (dihitung dari teks MENTAH)
# ---------------------------------------------------------------------------
KEYWORDS = {
    "kw_hadiah": r"selamat|hadiah|menang|pemenang|undian|terpilih|cek tunai|mobil|beruntung",
    "kw_uang": r"transfer|\btf\b|rekening|\brek\b|kirim|pulsa|saldo|bayar|biaya|tagihan|pinjam",
    "kw_aksi": r"klik|kunjungi|buka|download|unduh|install|instal|cek|hubungi|balas|ketik",
    "kw_rahasia": r"\bpin\b|kode|otp|verifikasi|password|sandi",
    "kw_promo": r"promo|diskon|bonus|paket|kuota|gratis|hemat|cashback|harga|beli|aktifkan",
    "kw_mendesak": r"segera|sekarang|hari ini|batas|terakhir|blokir|hangus|darurat|penting",
}
KEYWORD_RE = {k: re.compile(v, re.IGNORECASE) for k, v in KEYWORDS.items()}

META_FEATURES = [
    "panjang_log", "rasio_kapital", "rasio_angka", "ada_link", "ada_apk",
    "ada_nomor_hp", "ada_ussd", "jumlah_seru", *KEYWORDS.keys(),
]


def _features_one(text):
    t = str(text)
    n = max(len(t), 1)
    letters = [c for c in t if c.isalpha()]
    feats = [
        math.log1p(len(t)),
        sum(c.isupper() for c in letters) / max(len(letters), 1),
        sum(c.isdigit() for c in t) / n,
        float(bool(RE_URL.search(t))),
        float(bool(RE_APK.search(t))),
        float(bool(RE_PHONE.search(t))),
        float(bool(RE_USSD.search(t))),
        min(t.count("!"), 5),
    ]
    feats += [len(rx.findall(t)) for rx in KEYWORD_RE.values()]
    return feats


def extract_features(texts):
    return np.asarray([_features_one(t) for t in texts], dtype=float)


def meta_feature_names(transformer=None, input_features=None):
    return np.asarray(META_FEATURES, dtype=object)


# ---------------------------------------------------------------------------
# 3. Aturan keputusan (threshold khusus kelas penipuan)
# ---------------------------------------------------------------------------
def decide(proba_row, classes, threshold, fraud_label=1):
    """Penipuan kalau P(penipuan) >= threshold; selain itu ambil kelas
    lain (normal/promo) dengan probabilitas tertinggi."""
    classes = list(classes)
    i = classes.index(fraud_label)
    if proba_row[i] >= threshold:
        return fraud_label
    others = [j for j in range(len(classes)) if j != i]
    return classes[max(others, key=lambda j: proba_row[j])]


def decide_batch(proba, classes, threshold, fraud_label=1):
    return np.asarray([decide(p, classes, threshold, fraud_label) for p in proba])


# ---------------------------------------------------------------------------
# 4. Penjelasan (alasan) untuk pengguna bot
# ---------------------------------------------------------------------------
RULE_REASONS = [
    (lambda t: RE_APK.search(t), "Ada file/aplikasi .apk. Jangan dibuka atau di-install."),
    (lambda t: RE_URL.search(t), "Berisi link. Cek dulu alamat situsnya sebelum diklik."),
    (lambda t: KEYWORD_RE["kw_hadiah"].search(t), "Mengaku kamu menang hadiah atau undian."),
    (lambda t: KEYWORD_RE["kw_uang"].search(t), "Menyebut transfer, pulsa, rekening, atau biaya."),
    (lambda t: KEYWORD_RE["kw_rahasia"].search(t), "Menyebut PIN/kode/OTP. Jangan pernah bagikan ke siapa pun."),
    (lambda t: KEYWORD_RE["kw_mendesak"].search(t), "Memakai bahasa yang mendesak atau mengancam."),
]

META_LABEL = {
    "ada_link": "[ada link]", "ada_apk": "[ada .apk]", "ada_nomor_hp": "[ada nomor HP]",
    "ada_ussd": "[ada kode USSD]", "kw_hadiah": "[kata hadiah]", "kw_uang": "[kata uang/transfer]",
    "kw_aksi": "[ajakan klik/hubungi]", "kw_rahasia": "[kata PIN/OTP]", "kw_promo": "[kata promo]",
    "kw_mendesak": "[kata mendesak]", "jumlah_seru": "[banyak tanda seru]", "rasio_kapital": "[banyak huruf kapital]",
}


def rule_reasons(text):
    t = str(text)
    return [msg for check, msg in RULE_REASONS if check(t)]


def explain_tokens(pipeline, text, label, top_k=5):
    """Fitur kata/meta yang paling mendorong prediksi ke `label`.
    Bekerja untuk model linear (coef_) dan Multinomial NB."""
    clf = pipeline.steps[-1][1]
    feats = pipeline[:-1]
    classes = list(clf.classes_)
    row = classes.index(label)
    if hasattr(clf, "coef_"):
        weights = clf.coef_[row]
    elif hasattr(clf, "feature_log_prob_"):
        flp = clf.feature_log_prob_
        others = [j for j in range(len(classes)) if j != row]
        weights = flp[row] - flp[others].mean(axis=0)
    else:
        return []
    X = feats.transform([text])
    X = X.toarray()[0] if hasattr(X, "toarray") else np.asarray(X)[0]
    names = feats.get_feature_names_out()
    contrib = X * weights
    out = []
    for idx in np.argsort(-contrib):
        if contrib[idx] <= 0 or len(out) >= top_k:
            break
        name = str(names[idx])
        prefix, _, base = name.rpartition("__")
        if prefix == "char":  # n-gram karakter sulit dibaca manusia
            continue
        if prefix == "meta":
            base = META_LABEL.get(base)
            if base is None:
                continue
        else:
            base = " ".join(TOKEN_LABEL.get(w, w) for w in base.split())
        if base not in out:
            out.append(base)
    return out


def analyze(bundle, text):
    pipe = bundle["pipeline"]
    labels = bundle["labels"]
    fraud = bundle.get("fraud_label", 1)
    proba = pipe.predict_proba([text])[0]
    classes = [int(c) for c in pipe.classes_]
    label = int(decide(proba, classes, bundle["threshold"], fraud))
    oleh_aturan = False
    if bundle.get("rule_override", False) and label != fraud and RE_APK.search(str(text)):
        label, oleh_aturan = fraud, True
    return {
        "label": label,
        "oleh_aturan": oleh_aturan,
        "label_name": labels[label],
        "proba": {labels[c]: float(p) for c, p in zip(classes, proba)},
        "alasan": rule_reasons(text),
        "kata_kunci": explain_tokens(pipe, text, label),
    }


STATUS = {
    "penipuan": "🚨 MENCURIGAKAN: kemungkinan PENIPUAN",
    "promo": "📢 PROMO/IKLAN (biasanya tidak berbahaya)",
    "normal": "✅ Tampak AMAN (pesan biasa)",
}


def format_reply(result):
    lines = [STATUS.get(result["label_name"], result["label_name"])]
    if result.get("oleh_aturan"):
        lines.append("(ditandai oleh aturan keamanan: pesan berisi file .apk)")
    lines.append("")
    prob = " · ".join(f"{k} {v:.0%}" for k, v in result["proba"].items())
    lines.append(f"Keyakinan model: {prob}")
    if result["alasan"]:
        lines += ["", "Alasan:" if result["label_name"] == "penipuan" else "Catatan:"]
        lines += [f"• {r}" for r in result["alasan"]]
    if result["kata_kunci"]:
        lines += ["", "Paling berpengaruh: " + ", ".join(result["kata_kunci"])]
    lines += ["", "Ini prediksi otomatis, bukan jaminan. Jangan pernah membagikan PIN/OTP "
                  "atau meng-install file .apk dari pesan yang tidak jelas."]
    return "\n".join(lines)

In [ ]:
import importlib
import random
import re
import time
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from scipy.sparse.csgraph import connected_components
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             f1_score, fbeta_score, precision_score, recall_score)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import FeatureUnion, Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, MaxAbsScaler
from sklearn.svm import LinearSVC

import sms_utils
importlib.reload(sms_utils)  # supaya perubahan di sel %%writefile langsung terpakai

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = Path("data/dataset_sms_spam_v1.csv")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "sms_scam_model.joblib"

LABELS = {0: "normal", 1: "penipuan", 2: "promo"}
FRAUD = 1
TARGET_NAMES = [LABELS[i] for i in sorted(LABELS)]

pd.set_option("display.max_colwidth", 120)
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

## 2. Collect: memuat data
Dataset utama adalah data Wibisono (1.143 SMS). Kalau kamu sudah punya data kumpulan sendiri (modus baru: APK undangan, resi kurir, dan sejenisnya), simpan dengan format kolom yang sama (`Teks`, `label`) lalu masukkan path-nya ke `EXTRA_DATA`.

In [ ]:
def load_dataset(path, sumber):
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(path, encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    df.columns = [str(c).strip() for c in df.columns]
    text_col = next(c for c in df.columns if c.lower() in ("teks", "text", "sms", "pesan"))
    label_col = next(c for c in df.columns if c.lower() in ("label", "kelas", "class"))
    df = df.rename(columns={text_col: "teks", label_col: "label"})[["teks", "label"]]
    df = df.dropna(subset=["teks", "label"])
    df["teks"] = df["teks"].astype(str).str.strip()
    df = df[df["teks"].str.len() > 0]
    df["label"] = df["label"].astype(int)
    df["sumber"] = sumber
    return df.reset_index(drop=True)


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"File {DATA_PATH} belum ada. Unduh dari http://bit.ly/yw_sms_spam_indonesia "
        f"lalu simpan sebagai {DATA_PATH}"
    )

EXTRA_DATA = []  # contoh: [Path("data/penipuan_baru.csv")]

frames = [load_dataset(DATA_PATH, "wibisono")]
frames += [load_dataset(p, p.stem) for p in EXTRA_DATA]
df = pd.concat(frames, ignore_index=True)

assert set(df["label"].unique()) <= set(LABELS), f"Label tidak dikenal: {df['label'].unique()}"
print("Jumlah data:", len(df))
print(df["label"].map(LABELS).value_counts())
df.sample(5, random_state=SEED)

## 3. Exploratory Data Analysis

In [ ]:
df["n_kata"] = df["teks"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
df["label"].map(LABELS).value_counts().reindex(TARGET_NAMES).plot.bar(ax=axes[0], color=["#4C78A8", "#E45756", "#F2A541"])
axes[0].set_title("Jumlah pesan per kelas")
axes[0].tick_params(axis="x", rotation=0)

data_box = [df.loc[df["label"] == k, "n_kata"] for k in sorted(LABELS)]
axes[1].boxplot(data_box)
axes[1].set_xticks(range(1, len(TARGET_NAMES) + 1), TARGET_NAMES)
axes[1].set_title("Panjang pesan (jumlah kata)")
plt.tight_layout()
plt.show()

print(df.groupby(df["label"].map(LABELS))["n_kata"].describe().round(1))

In [ ]:
# Kata paling sering per kelas (setelah normalisasi, tanpa stopword)
STOPWORDS_ID = """yang dan di ke dari ini itu untuk dengan pada ada saya aku kamu anda kita kami
ya yg aja saja juga sudah belum tidak bisa akan atau karena jadi kalau lagi mau sama dalam oleh
nya lah kah pun dong deh sih kok nih tuh si para buat tapi biar""".split()

for k, name in LABELS.items():
    cv = CountVectorizer(preprocessor=sms_utils.normalize_text, stop_words=STOPWORDS_ID)
    X = cv.fit_transform(df.loc[df["label"] == k, "teks"])
    freq = np.asarray(X.sum(axis=0)).ravel()
    top = [cv.get_feature_names_out()[i] for i in freq.argsort()[::-1][:15]]
    print(f"{name:9s}: {', '.join(top)}")

## 4. Preprocessing
`normalize_text` melakukan:
1. Lowercase dan normalisasi unicode.
2. Masking: URL → `urltoken`, nomor HP → `nomortoken`, kode USSD (`*123#`) → `ussdtoken`, token yang mayoritas angka → `angkatoken`. Template penipuan sering sama persis dan hanya beda nomor atau link, jadi masking membuat model belajar polanya, bukan menghafal nomornya.
3. Hapus tanda baca dan rapikan huruf berulang (`gratisss` → `gratiss`).
4. Normalisasi slang/singkatan (`yg` → `yang`, `tf` → `transfer`, dan seterusnya).

Kata "leet" seperti `h4diah`/`lnfo` **sengaja tidak dinormalisasi**, karena itu ciri khas penipu.

In [ ]:
df["teks_norm"] = df["teks"].map(sms_utils.normalize_text)

for _, r in df.groupby("label").head(2).iterrows():
    print(f"[{LABELS[r.label]}] {r.teks}")
    print(f"   -> {r.teks_norm}\n")

In [ ]:
# Fitur buatan sendiri (dihitung dari teks mentah)
meta = pd.DataFrame(sms_utils.extract_features(df["teks"]), columns=sms_utils.META_FEATURES)
meta["kelas"] = df["label"].map(LABELS)
meta.groupby("kelas").mean().T.round(3)

## 5. Deduplikasi dan pengelompokan near-duplicate (anti data leakage)
Pesan penipuan banyak berupa **template yang diulang**. Kalau satu varian masuk train dan varian lain masuk test, skor model akan terlihat bagus padahal model hanya menghafal. Karena itu:
1. Duplikat persis (setelah normalisasi) dibuang.
2. Pesan yang sangat mirip (cosine similarity n-gram karakter ≥ `NEAR_DUP_THRESHOLD`) dimasukkan ke **grup yang sama**, dan satu grup tidak boleh terpecah antar-split.

In [ ]:
NEAR_DUP_THRESHOLD = 0.80

n_awal = len(df)
konflik = df.groupby("teks_norm")["label"].nunique()
konflik = konflik[konflik > 1]
if len(konflik):
    print(f"⚠️ {len(konflik)} teks identik dengan label BERBEDA. Cek manual:")
    print(df[df["teks_norm"].isin(konflik.index)].sort_values("teks_norm")[["teks", "label"]].head(20))

df = df.drop_duplicates("teks_norm").reset_index(drop=True)
print(f"Duplikat persis dibuang: {n_awal - len(df)} → sisa {len(df)} pesan")

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit_transform(df["teks_norm"])
sim = (vec @ vec.T).tocsr()
sim.data[sim.data < NEAR_DUP_THRESHOLD] = 0
sim.eliminate_zeros()
n_grup, df["group"] = connected_components(sim, directed=False)

ukuran = df["group"].value_counts()
print(f"Jumlah grup: {n_grup} | grup berisi >1 pesan: {(ukuran > 1).sum()} | grup terbesar: {ukuran.max()} pesan")
contoh = ukuran[ukuran > 1].index[:2]
for g in contoh:
    print("\nContoh grup near-duplicate:")
    print(df.loc[df["group"] == g, "teks"].head(3).to_string())

## 6. Split train / validation / test
Split memakai `StratifiedGroupKFold`: proporsi kelas terjaga dan grup near-duplicate tidak bocor. Hasilnya kira-kira **72% train / 14% validation / 14% test**.
- **Validation** dipakai untuk memilih model dan threshold.
- **Test** hanya dipakai **sekali** di akhir.

In [ ]:
def group_split(frame, n_splits, seed):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rest_idx, hold_idx = next(sgkf.split(frame, frame["label"], frame["group"]))
    return frame.iloc[rest_idx].reset_index(drop=True), frame.iloc[hold_idx].reset_index(drop=True)


train_full, test_df = group_split(df, n_splits=7, seed=SEED)
train_df, val_df = group_split(train_full, n_splits=6, seed=SEED)

assert not set(train_df["group"]) & set(val_df["group"])
assert not set(train_full["group"]) & set(test_df["group"])

ringkas = pd.DataFrame({
    nama: d["label"].map(LABELS).value_counts().reindex(TARGET_NAMES)
    for nama, d in [("train", train_df), ("val", val_df), ("test", test_df)]
})
ringkas.loc["TOTAL"] = ringkas.sum()
ringkas

## 7. Augmentasi (hanya untuk train)
Augmentasi hanya diterapkan pada kelas **penipuan** dan **promo** untuk menyeimbangkan kelas. Tekniknya:
- **Ganti entitas:** operator, e-commerce, bank, kurir, dan jenis hadiah ditukar dengan yang sejenis.
- **Singkatan/alay:** `dengan` → `dgn`, `selamat` → `slmt`.
- **Leet ringan:** `info` → `lnfo`, `hadiah` → `h4diah`.
- **Word dropout** dan **huruf kapital semua**.

Setiap baris hasil augmentasi diberi `is_synthetic = 1`. Validation dan test **tidak** diaugmentasi.

In [ ]:
ENTITAS = [
    ["Telkomsel", "Indosat", "IM3", "Tri", "XL", "Axis", "Smartfren"],
    ["Lazada", "Shopee", "Tokopedia", "Blibli", "Bukalapak"],
    ["BRI", "BCA", "Mandiri", "BNI", "BSI"],
    ["JNE", "J&T", "SiCepat", "AnterAja", "Ninja Xpress"],
    ["mobil", "motor", "laptop", "emas", "rumah"],
]
ENTITAS_RE = [(re.compile(r"\b(" + "|".join(map(re.escape, grp)) + r")\b", re.IGNORECASE), grp) for grp in ENTITAS]
SINGKAT = {"dengan": "dgn", "yang": "yg", "untuk": "utk", "tidak": "tdk", "sudah": "sdh", "selamat": "slmt",
           "mendapatkan": "mendptkan", "informasi": "info", "hubungi": "hub", "nomor": "no",
           "sekarang": "skrg", "rekening": "rek", "transfer": "tf"}
LEET = {"i": "l", "o": "0", "a": "4", "e": "3", "s": "5"}
RE_JANGAN_DIUBAH = re.compile(r"https?://|www\.|\.(com|net|id|ly|xyz|apk)\b|\d", re.IGNORECASE)


def ganti_entitas(t, rng):
    for rx, grp in ENTITAS_RE:
        t = rx.sub(lambda m: rng.choice(grp), t)
    return t


def singkat(t, rng, p=0.5):
    return " ".join(SINGKAT.get(w.lower(), w) if rng.random() < p else w for w in t.split())


def leet(t, rng, p=0.04):
    out = []
    for w in t.split():
        if RE_JANGAN_DIUBAH.search(w):  # jangan rusak URL/angka
            out.append(w)
        else:
            out.append("".join(LEET[c] if c in LEET and rng.random() < p else c for c in w))
    return " ".join(out)


def dropout(t, rng, p=0.1):
    kata = t.split()
    if len(kata) <= 5:
        return t
    sisa = [w for w in kata if RE_JANGAN_DIUBAH.search(w) or rng.random() > p]
    return " ".join(sisa) if sisa else t


def kapital(t, rng):
    return t.upper()


OPS = [singkat, leet, dropout, kapital]


def augment_text(t, rng):
    t = ganti_entitas(t, rng)
    k = rng.choice([1, 1, 2])
    for op in rng.sample(OPS, k):
        t = op(t, rng)
    return t


def augment_train(train, kelas=(1, 2), seed=SEED, max_tries=5):
    rng = random.Random(seed)
    counts = train["label"].value_counts()
    target = counts.max()
    seen = set(train["teks"].map(sms_utils.normalize_text))
    baru = []
    for k in kelas:
        sumber = train[train["label"] == k]
        perlu = target - counts.get(k, 0)
        tries = 0
        while perlu > 0 and tries < perlu * max_tries + 50:
            tries += 1
            asal = sumber.iloc[rng.randrange(len(sumber))]
            t = augment_text(asal["teks"], rng)
            norm = sms_utils.normalize_text(t)
            if norm in seen:
                continue
            seen.add(norm)
            baru.append({"teks": t, "label": k, "sumber": "augmentasi", "group": asal["group"], "is_synthetic": 1})
            perlu -= 1
    out = pd.concat([train.assign(is_synthetic=0), pd.DataFrame(baru)], ignore_index=True)
    return out.sample(frac=1, random_state=seed).reset_index(drop=True)


train_aug = augment_train(train_df)
print("Train asli      :", train_df["label"].map(LABELS).value_counts().reindex(TARGET_NAMES).to_dict())
print("Train + augment :", train_aug["label"].map(LABELS).value_counts().reindex(TARGET_NAMES).to_dict())
train_aug[train_aug["is_synthetic"] == 1].head(8)[["teks", "label"]]

## 8. Feature engineering dan training
Fitur gabungan (`FeatureUnion`):
- **TF-IDF kata** (unigram + bigram) dari teks ternormalisasi.
- **TF-IDF n-gram karakter** (3–5), tahan terhadap typo dan leet.
- **Fitur buatan** (ada link, ada .apk, kata hadiah/transfer/OTP, rasio kapital, dan lainnya), diskalakan dengan `MaxAbsScaler`.

Kandidat model: Multinomial Naive Bayes (baseline), Logistic Regression, dan Linear SVM. Masing-masing dilatih dengan **data asli** dan **data asli + augmentasi**, lalu dibandingkan di validation set. Fokus utamanya adalah **recall kelas penipuan**: lebih baik salah curiga daripada kecolongan.

In [ ]:
def build_features():
    return FeatureUnion([
        ("word", TfidfVectorizer(preprocessor=sms_utils.normalize_text, ngram_range=(1, 2),
                                 sublinear_tf=True, min_df=1)),
        ("char", TfidfVectorizer(preprocessor=sms_utils.normalize_text, analyzer="char_wb",
                                 ngram_range=(3, 5), sublinear_tf=True, min_df=2)),
        ("meta", make_pipeline(
            FunctionTransformer(sms_utils.extract_features, feature_names_out=sms_utils.meta_feature_names),
            MaxAbsScaler())),
    ])


CANDIDATES = {
    "NaiveBayes": lambda: Pipeline([
        ("tfidf", TfidfVectorizer(preprocessor=sms_utils.normalize_text, ngram_range=(1, 2), sublinear_tf=True)),
        ("clf", MultinomialNB(alpha=0.1)),
    ]),
    "LogReg_C1": lambda: Pipeline([("features", build_features()),
                                   ("clf", LogisticRegression(C=1, class_weight="balanced", max_iter=3000))]),
    "LogReg_C5": lambda: Pipeline([("features", build_features()),
                                   ("clf", LogisticRegression(C=5, class_weight="balanced", max_iter=3000))]),
    "LogReg_C20": lambda: Pipeline([("features", build_features()),
                                    ("clf", LogisticRegression(C=20, class_weight="balanced", max_iter=3000))]),
    "LinearSVM": lambda: Pipeline([("features", build_features()),
                                   ("clf", LinearSVC(C=0.5, class_weight="balanced"))]),
}


def skor(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "recall_penipuan": recall_score(y_true, y_pred, labels=[FRAUD], average="macro", zero_division=0),
        "precision_penipuan": precision_score(y_true, y_pred, labels=[FRAUD], average="macro", zero_division=0),
        "f1_penipuan": f1_score(y_true, y_pred, labels=[FRAUD], average="macro", zero_division=0),
    }


TRAIN_SETS = {"asli": train_df, "asli+augmentasi": train_aug}
hasil, fitted = [], {}
for data_name, tr in TRAIN_SETS.items():
    for model_name, factory in CANDIDATES.items():
        t0 = time.time()
        model = factory().fit(tr["teks"], tr["label"])
        pred = model.predict(val_df["teks"])
        fitted[(model_name, data_name)] = model
        hasil.append({"model": model_name, "data_train": data_name, **skor(val_df["label"], pred),
                      "detik": round(time.time() - t0, 1)})

hasil_df = pd.DataFrame(hasil).sort_values(["macro_f1", "recall_penipuan"], ascending=False).reset_index(drop=True)
hasil_df.style.format(precision=3).background_gradient(subset=["macro_f1", "recall_penipuan"], cmap="Greens")

## 9. Pilih model dan atur threshold penipuan
Bot butuh probabilitas, jadi dipilih model terbaik yang punya `predict_proba` (Naive Bayes atau Logistic Regression). Setelah itu, threshold P(penipuan) dicari di validation set dengan memaksimalkan **F2-score**. F2 memberi bobot recall dua kali lebih besar dari precision.

In [ ]:
punya_proba = hasil_df[hasil_df["model"].map(lambda m: hasattr(CANDIDATES[m](), "predict_proba"))]
best_row = punya_proba.iloc[0]
BEST_MODEL, BEST_DATA = best_row["model"], best_row["data_train"]
best_model = fitted[(BEST_MODEL, BEST_DATA)]
print(f"Model terpilih: {BEST_MODEL} (data train: {BEST_DATA})")

classes = list(best_model.classes_)
proba_val = best_model.predict_proba(val_df["teks"])
# Rentang dibatasi maksimal 0.60 karena tujuan kita mengutamakan recall penipuan
thresholds = np.round(np.arange(0.15, 0.61, 0.01), 2)
kurva = []
for thr in thresholds:
    pred = sms_utils.decide_batch(proba_val, classes, thr, FRAUD)
    y_bin, p_bin = (val_df["label"] == FRAUD), (pred == FRAUD)
    kurva.append({"threshold": thr,
                  "precision": precision_score(y_bin, p_bin, zero_division=0),
                  "recall": recall_score(y_bin, p_bin, zero_division=0),
                  "f2": fbeta_score(y_bin, p_bin, beta=2, zero_division=0),
                  "macro_f1": f1_score(val_df["label"], pred, average="macro")})
kurva = pd.DataFrame(kurva)
# Ambil F2 tertinggi. Kalau banyak threshold seri (sering terjadi di validation set kecil),
# pilih nilai tengahnya supaya tidak menempel di ujung rentang.
terbaik = kurva[np.isclose(kurva["f2"], kurva["f2"].max())]["threshold"]
THRESHOLD = float(np.round(terbaik.median(), 2))
print(f"Threshold penipuan terpilih: {THRESHOLD:.2f}")
print(kurva[kurva["threshold"].isin([0.3, 0.4, 0.5, THRESHOLD])].round(3).to_string(index=False))

plt.figure(figsize=(7, 3.5))
for col in ["precision", "recall", "f2"]:
    plt.plot(kurva["threshold"], kurva[col], label=col)
plt.axvline(THRESHOLD, ls="--", c="gray")
plt.xlabel("threshold P(penipuan)")
plt.title("Kelas penipuan di validation set")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Latih ulang dengan train + validation, lalu evaluasi di test set
Model dilatih ulang memakai seluruh data non-test (train + validation, ditambah augmentasi kalau model terpilih memakainya). Test set dievaluasi **sekali**.

In [ ]:
final_train = augment_train(train_full) if BEST_DATA == "asli+augmentasi" else train_full
final_model = CANDIDATES[BEST_MODEL]().fit(final_train["teks"], final_train["label"])

proba_test = final_model.predict_proba(test_df["teks"])
pred_argmax = final_model.classes_[proba_test.argmax(axis=1)]
pred_test = sms_utils.decide_batch(proba_test, list(final_model.classes_), THRESHOLD, FRAUD)

print(f"=== {BEST_MODEL} | threshold {THRESHOLD:.2f} ===")
print(classification_report(test_df["label"], pred_test, target_names=TARGET_NAMES, digits=3))
banding = pd.DataFrame({"argmax (0.5)": skor(test_df["label"], pred_argmax),
                        f"threshold {THRESHOLD:.2f}": skor(test_df["label"], pred_test)}).round(3)
print(banding)

fig, ax = plt.subplots(figsize=(4.6, 4))
ConfusionMatrixDisplay.from_predictions(test_df["label"], pred_test, display_labels=TARGET_NAMES,
                                        cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Confusion matrix (test)")
plt.tight_layout()
plt.show()
METRICS_TEST = skor(test_df["label"], pred_test)

In [ ]:
# Analisis kesalahan: pesan mana yang salah diprediksi?
err = test_df.assign(pred=pred_test, p_penipuan=proba_test[:, list(final_model.classes_).index(FRAUD)])
err = err[err["label"] != err["pred"]]
err["label"], err["pred"] = err["label"].map(LABELS), err["pred"].map(LABELS)
print(f"{len(err)} kesalahan dari {len(test_df)} pesan test")
err[["teks", "label", "pred", "p_penipuan"]].sort_values("p_penipuan", ascending=False).round(3)

## 11. Interpretasi: kata apa yang paling menentukan?
Bagian ini menampilkan kata-kata yang paling kuat mendorong prediksi ke tiap kelas, lalu menguji model dengan beberapa pesan modus terbaru yang **tidak ada** di dataset.

Bot memakai sistem **hibrida**: prediksi model ditambah satu aturan keamanan (`rule_override`). Pesan yang berisi file `.apk` selalu ditandai mencurigakan, karena modus ini belum terwakili di data latih. Kolom `aturan` di output menunjukkan kapan aturan itu yang memutuskan. Evaluasi di bagian 10 murni dari model, tanpa aturan.

In [ ]:
clf = final_model.steps[-1][1]
names = final_model[:-1].get_feature_names_out()
for i, c in enumerate(clf.classes_):
    if hasattr(clf, "coef_"):
        w = clf.coef_[i]
    else:  # Naive Bayes
        others = [j for j in range(len(clf.classes_)) if j != i]
        w = clf.feature_log_prob_[i] - clf.feature_log_prob_[others].mean(axis=0)
    mask = np.array([not str(n).startswith("char__") for n in names])
    idx = np.where(mask)[0][np.argsort(-w[mask])[:15]]
    print(f"{LABELS[int(c)]:9s}: {', '.join(str(names[j]).split('__')[-1] for j in idx)}")

In [ ]:
BUNDLE = {
    "pipeline": final_model,
    "threshold": THRESHOLD,
    "labels": LABELS,
    "fraud_label": FRAUD,
    # Aturan keamanan di bot: pesan berisi .apk SELALU ditandai mencurigakan,
    # walaupun model berkata lain (modus APK belum ada di dataset Wibisono).
    "rule_override": True,
}

PESAN_UJI = [
    "Assalamualaikum, ini undangan pernikahan kami. Mohon dibuka ya: Undangan_Pernikahan.apk",
    "Paket Anda dari J&T tertahan karena alamat tidak lengkap. Lihat foto resi di sini: Resi_JNT.apk",
    "SELAMAT! No Anda terpilih mendapat cek tunai Rp175jt dari Shopee. Info klik bit.ly/hadiah-shopee",
    "Mah ini aku ganti nomor. Tolong transfer 500rb dulu ke rek BRI ini ya, penting banget",
    "Bro nanti malam jadi futsal jam 8? kabarin ya",
    "Nikmati kuota 30GB cuma Rp50rb! Aktifkan sekarang di *363#",
    "Kode OTP Anda 482913. JANGAN berikan kode ini kepada siapapun termasuk pihak bank.",
    "nama1, rapat kelompok besok jam 10 di perpus ya",
]
for t in PESAN_UJI:
    r = sms_utils.analyze(BUNDLE, t)
    prob = ", ".join(f"{k} {v:.0%}" for k, v in r["proba"].items())
    tanda = " (aturan .apk)" if r["oleh_aturan"] else ""
    print(f"{r['label_name'].upper():9s}{tanda} | {prob}\n   {t}\n   kata kunci: {r['kata_kunci']}\n")

In [ ]:
# Contoh balasan bot untuk satu pesan
print(sms_utils.format_reply(sms_utils.analyze(BUNDLE, PESAN_UJI[0])))

## 12. Simpan model

In [ ]:
BUNDLE.update({
    "model_name": BEST_MODEL,
    "data_train": BEST_DATA,
    "metrics_test": METRICS_TEST,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "sklearn_version": sklearn.__version__,
    "sumber_data": "Rahmi, F. and Wibisono, Y. (2016). Aplikasi SMS Spam Filtering pada Android "
                   "menggunakan Naive Bayes, Unpublished manuscript.",
    "lisensi_data": "CC BY-SA 4.0",
})
joblib.dump(BUNDLE, MODEL_PATH)
print(f"Tersimpan: {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1024:.0f} KB)")

# Uji load ulang, persis seperti yang dilakukan bot
loaded = joblib.load(MODEL_PATH)
print(sms_utils.analyze(loaded, "Selamat anda menang undian, kirim pulsa 100rb untuk klaim")["label_name"])

## 13. (Opsional) Fine-tune IndoBERT sebagai pembanding
Bagian ini butuh `torch`, `transformers`, dan `accelerate`. Idealnya dijalankan dengan GPU; di CPU bisa memakan waktu cukup lama. Pasang dulu:
`pip install torch transformers accelerate`, lalu ubah `RUN_INDOBERT = True`.

IndoBERT dilatih dengan `train_df` dan dievaluasi di validation tiap epoch, lalu dinilai di **test set yang sama** dengan model klasik supaya perbandingannya adil. Catatan: model klasik di bagian 10 dilatih dengan train + validation, jadi IndoBERT di sini melihat data latih sedikit lebih sedikit.

In [ ]:
RUN_INDOBERT = False

if RUN_INDOBERT:
    import torch
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                              DataCollatorWithPadding, Trainer, TrainingArguments)

    MODEL_ID = "indobenchmark/indobert-base-p1"
    tok = AutoTokenizer.from_pretrained(MODEL_ID)

    class SMSDataset(torch.utils.data.Dataset):
        def __init__(self, frame):
            self.enc = tok(list(frame["teks"].map(sms_utils.normalize_text)), truncation=True, max_length=128)
            self.labels = list(frame["label"].astype(int))

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, i):
            item = {k: v[i] for k, v in self.enc.items()}
            item["labels"] = self.labels[i]
            return item

    def compute_metrics(p):
        preds = p.predictions.argmax(-1)
        return {"macro_f1": f1_score(p.label_ids, preds, average="macro"),
                "recall_penipuan": recall_score(p.label_ids, preds, labels=[FRAUD], average="macro", zero_division=0)}

    bert = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(LABELS), id2label=LABELS, label2id={v: k for k, v in LABELS.items()})
    args = TrainingArguments(
        output_dir="models/indobert", num_train_epochs=4, learning_rate=3e-5, weight_decay=0.01,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="macro_f1", save_total_limit=1, logging_steps=20,
        report_to="none", seed=SEED)
    trainer = Trainer(model=bert, args=args, train_dataset=SMSDataset(train_df),
                      eval_dataset=SMSDataset(val_df), data_collator=DataCollatorWithPadding(tok),
                      compute_metrics=compute_metrics)
    trainer.train()

    pred_bert = trainer.predict(SMSDataset(test_df)).predictions.argmax(-1)
    print(classification_report(test_df["label"], pred_bert, target_names=TARGET_NAMES, digits=3))
    print(pd.DataFrame({BEST_MODEL: METRICS_TEST, "IndoBERT": skor(test_df["label"], pred_bert)}).round(3))
else:
    print("Dilewati. Ubah RUN_INDOBERT = True untuk menjalankan.")

## 14. Deploy: bot Telegram
Dua sel di bawah menulis `requirements.txt` dan `bot.py`. **Jangan jalankan bot di dalam notebook**, karena bot berjalan terus (blocking). Jalankan dari terminal VS Code:

1. Di Telegram, chat **@BotFather**, kirim `/newbot`, lalu salin token yang diberikan.
2. Set token-nya:
   - PowerShell: `$env:TELEGRAM_BOT_TOKEN="123456:ABC..."`
   - Linux/Mac: `export TELEGRAM_BOT_TOKEN="123456:ABC..."`
3. Jalankan `python bot.py`, lalu forward pesan mencurigakan ke bot-mu.

**Penting:** `bot.py`, `sms_utils.py`, dan folder `models/` harus berada di satu folder. Versi scikit-learn di tempat bot dijalankan juga harus sama dengan versi saat training.

In [ ]:
%%writefile requirements.txt
pandas>=2.0
numpy>=1.24
scipy>=1.10
scikit-learn>=1.3
matplotlib>=3.7
joblib>=1.3
python-telegram-bot>=21.0
ipykernel
# Opsional (bagian IndoBERT):
# torch
# transformers>=4.46
# accelerate

In [ ]:
%%writefile bot.py
"""Bot Telegram detektor pesan penipuan SMS/WA.

Cara pakai:
  1. Buat bot lewat @BotFather di Telegram, salin token-nya.
  2. Set environment variable TELEGRAM_BOT_TOKEN.
       PowerShell : $env:TELEGRAM_BOT_TOKEN="123:ABC..."
       Linux/Mac  : export TELEGRAM_BOT_TOKEN="123:ABC..."
  3. Jalankan dari folder proyek:  python bot.py
"""
import logging
import os
from pathlib import Path

import joblib
from telegram import Update
from telegram.ext import Application, CommandHandler, ContextTypes, MessageHandler, filters

import sms_utils  # wajib ada di folder yang sama (dipakai di dalam pipeline model)

MODEL_PATH = Path(os.environ.get("MODEL_PATH", "models/sms_scam_model.joblib"))
BUNDLE = joblib.load(MODEL_PATH)

logging.basicConfig(format="%(asctime)s %(levelname)s %(message)s", level=logging.INFO)
logging.getLogger("httpx").setLevel(logging.WARNING)
log = logging.getLogger("bot")

WELCOME = (
    "Halo! 👋\n\n"
    "Forward atau tempel pesan SMS/WhatsApp yang mencurigakan ke sini. "
    "Saya akan cek apakah pesan itu kemungkinan PENIPUAN, PROMO, atau pesan biasa.\n\n"
    "Isi pesanmu tidak saya simpan."
)


async def start(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    await update.effective_message.reply_text(WELCOME)


async def cek_pesan(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    msg = update.effective_message
    text = msg.text or msg.caption
    if not text:
        await msg.reply_text("Kirim pesannya dalam bentuk teks, ya. Gambar/stiker belum bisa saya baca.")
        return
    result = sms_utils.analyze(BUNDLE, text)
    # Log hanya hasil prediksi, bukan isi pesan (privasi pengguna)
    log.info("prediksi=%s proba=%s", result["label_name"], {k: round(v, 3) for k, v in result["proba"].items()})
    await msg.reply_text(sms_utils.format_reply(result))


def main() -> None:
    token = os.environ.get("TELEGRAM_BOT_TOKEN")
    if not token:
        raise SystemExit("Set environment variable TELEGRAM_BOT_TOKEN dulu (token dari @BotFather).")
    app = Application.builder().token(token).build()
    app.add_handler(CommandHandler(["start", "help"], start))
    app.add_handler(MessageHandler((filters.TEXT | filters.CAPTION) & ~filters.COMMAND, cek_pesan))
    log.info("Bot berjalan. Tekan Ctrl+C untuk berhenti.")
    app.run_polling()


if __name__ == "__main__":
    main()

## 15. Langkah berikutnya
- **Tambah data modus baru** (APK undangan, resi kurir, tilang elektronik, lowongan kerja part-time, dan sejenisnya) ke `EXTRA_DATA`. Idealnya, test set juga diisi data baru yang asli supaya kemampuan model menangkap modus baru benar-benar teruji.
- **Tambah chat normal gaya WhatsApp** di kelas 0, karena gaya bahasa WA berbeda dengan SMS dan bot akan lebih banyak menerima pesan WA.
- **Masking nama di sisi bot:** di dataset, nama orang diganti `nama1`. Pertimbangkan NER atau aturan sederhana supaya nama di pesan masuk juga diganti `nama1`.
- **Monitoring:** catat distribusi prediksi per hari (tanpa isi pesan) untuk mendeteksi drift.
- Kalau dataset hasil augmentasi dipublikasikan, gunakan lisensi **CC BY-SA 4.0** dan cantumkan sitasi Rahmi & Wibisono (2016).